In [8]:
import pandas as pd
import plotly.graph_objects as go
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))



In [9]:
price_df_1 = pd.read_csv("data/prices_round_0_day_-2.csv" , sep = ';')
price_df_2 = pd.read_csv("data/prices_round_0_day_-1.csv" , sep = ';')

In [10]:
tomatoe_price_df_1 = price_df_1[price_df_1['product']=='TOMATOES'].reset_index(drop=True)
tomatoe_price_df_2 = price_df_2[price_df_2['product']=='TOMATOES'].reset_index(drop=True)

In [11]:
def mid_prices(df):
    df['mid_price'] = (df['bid_price_1'] + df['ask_price_1']) / 2
    df['wall_mid'] = (df['bid_price_2'] + df['ask_price_2']) / 2

mid_prices(tomatoe_price_df_1)
mid_prices(tomatoe_price_df_2)

In [12]:
from statsmodels.tsa.arima.model import ARIMA
import numpy as np

# List of your dataframes
data_list = [tomatoe_price_df_2, tomatoe_price_df_1]
def clean_arima(res, target_index):
    # 1. Convert the numpy array to a pandas Series using the original index
    # Note: If res is the model result object, use res.fittedvalues
    # If res is already the array, just use res
    data = res.fittedvalues if hasattr(res, 'fittedvalues') else res
    
    s = pd.Series(data, index=target_index)
    
    # 2. Set the first value to NaN
    s.iloc[0] = np.nan
    
    return s


for i, df in enumerate(data_list):
    # 1. Define the models for this specific DataFrame
    model_mid = ARIMA(df['mid_price'].values, order=(0, 1, 1)) 
    model_wall = ARIMA(df['wall_mid'].values, order=(0, 1, 1)) 

    # 2. Fit both models
    res_mid = model_mid.fit()
    mid_coef = res_mid.params

    res_wall = model_wall.fit()
    wall_coef = res_wall.params

    # 3. Print the Summaries
    print("\n" + "="*80)
    print(f"DATAFRAME {i} - MODEL 1: MID PRICE ARIMA(0,1,1)")
    print("="*80)
    print(res_mid.summary())

    print("\n" + "="*80)
    print(f"DATAFRAME {i} - MODEL 2: WALL PRICE ARIMA(0,1,1)")
    print("="*80)
    print(res_wall.summary())
    res_wall = model_wall.fit()
    wall_coef = res_wall.params

    # 4. Potential Alpha using coeffiecients from fast signal the mid_price on more stable signal wall_mid
    res_wall_reversed = model_wall.smooth(mid_coef)
    mid_price_reversed =model_mid.smooth(wall_coef)
    
    # 5. Add the results back to  dataframe for graphing later
    # Function to clean the first value and align

    df['ARIMA_mid'] = clean_arima(res_mid, df.index)
    df['ARIMA_wall'] = clean_arima(res_wall, df.index)
    df['ARIMA_wall_rever'] = clean_arima(res_wall_reversed, df.index)
    df['ARIMA_mid_rever'] = clean_arima(mid_price_reversed, df.index)
    


DATAFRAME 0 - MODEL 1: MID PRICE ARIMA(0,1,1)
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                10000
Model:                 ARIMA(0, 1, 1)   Log Likelihood              -15794.552
Date:                Sun, 12 Apr 2026   AIC                          31593.104
Time:                        19:15:54   BIC                          31607.525
Sample:                             0   HQIC                         31597.986
                              - 10000                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1         -0.5553      0.007    -82.459      0.000      -0.568      -0.542
sigma2         1.3789      0.011    121.209      0.000       1.357       1.401
Ljung

In [47]:
import numpy as np
def get_slope(y):
    x = np.arange(len(y))
    # Standard linear regression slope formula: 
    # slope = covariance(x, y) / variance(x)
    return np.polyfit(x, y, 1)[0]

dfs = [tomatoe_price_df_2, tomatoe_price_df_1]

for df in dfs:
    # Ensure the column exists to avoid KeyErrors
    # 1. Create a rolling window of 50 ticks
    # 2. Apply the slope function
    df['rolling_slope'] = (
        df['ARIMA_wall_rever']
        .rolling(window=50)
        .apply(get_slope, raw=True)
    )
    df.loc[df.index[:50], 'rolling_slope'] = np.nan


In [157]:
def get_mobi2(df):
    # Select volume columns for bids and asks
    bid_cols = ['bid_volume_1']
    ask_cols = ['ask_volume_1']
    
    # .sum(axis=1) treats NaNs as 0. 
    # .replace(0, np.nan) prevents division by zero if the book is totally empty
    total_bid_vol = df[bid_cols].sum(axis=1)
    total_ask_vol = df[ask_cols].sum(axis=1)
    
    # Calculate imbalance
    imbalance = (total_bid_vol - total_ask_vol) / (total_bid_vol + total_ask_vol)
    
    # Fill any remaining NaNs (caused by 0/0) with 0
    return imbalance.fillna(0)


def get_mobi3(df):
    # Select volume columns for bids and asks
    bid_cols = ['bid_volume_1', 'bid_volume_2', 'bid_volume_3']
    ask_cols = ['ask_volume_1', 'ask_volume_2', 'ask_volume_3']
    
    # .sum(axis=1) treats NaNs as 0. 
    # .replace(0, np.nan) prevents division by zero if the book is totally empty
    total_bid_vol = df[bid_cols].sum(axis=1)
    total_ask_vol = df[ask_cols].sum(axis=1)
    
    # Calculate imbalance
    imbalance = (total_bid_vol - total_ask_vol) / (total_bid_vol + total_ask_vol)
    
    # Fill any remaining NaNs (caused by 0/0) with 0
    return imbalance.fillna(0)

for df in dfs:
    df['obi3'] = get_mobi3(df).replace(0, np.nan)
    df['obi'] = get_mobi2(df).replace(0, np.nan)
    obi_present  = df['obi'].notna()
    obi3_present = df['obi3'].notna()
    only_obi3 = obi3_present & ~obi_present
    df_only_obi3 = df[only_obi3]
    both = obi_present & obi3_present
    df_both = df[both]
    neither = ~obi_present & ~obi3_present
    df_neither = df[neither]
    only_obi = obi_present & ~obi3_present
    df_only_obi = df[only_obi]

    df['signal'] = only_obi3.astype(int)
    df['obi3'] = df['obi3'].where(only_obi3)
    df['bid_price_3'] = df['bid_price_3'].where(only_obi3)
    df['ask_price_3'] = df['ask_price_3'].where(only_obi3)
    
    print("only obi3:", only_obi3.sum())
    print("only obi :", only_obi.sum())
    print("both     :", both.sum())
    print("neither  :", neither.sum())
    print(only_obi3.unique)
    




only obi3: 58
only obi : 0
both     : 664
neither  : 9278
<bound method Series.unique of 0       False
1       False
2       False
3       False
4       False
        ...  
9995    False
9996    False
9997    False
9998    False
9999    False
Length: 10000, dtype: bool>
only obi3: 64
only obi : 0
both     : 657
neither  : 9279
<bound method Series.unique of 0       False
1       False
2       False
3       False
4       False
        ...  
9995    False
9996    False
9997    False
9998    False
9999    False
Length: 10000, dtype: bool>


In [158]:

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create a figure with 2 rows
price_graph = make_subplots(
    rows=2, 
    cols=1, 
    specs=[[{"secondary_y": True}],  # Row 1
           [{"secondary_y": True}]]  # Row 2
)

for i, df in enumerate([tomatoe_price_df_2, tomatoe_price_df_1]):
    row = i + 1
    #price_graph.add_trace(go.Scatter(x=df.index, y=df['mid_price'], mode='lines', name='Mid Price'), row=row, col=1)
    #price_graph.add_trace(go.Scatter(x=df.index, y=df['wall_mid'], mode='lines', name='Wall Mid'), row=row, col=1)
    #price_graph.add_trace(go.Scatter(x=df.index, y=df['ARIMA_mid'], mode='lines', name='ARIMA Mid'), row=row, col=1)
    #price_graph.add_trace(go.Scatter(x=df.index, y=df['ARIMA_wall'], mode='lines', name='ARIMA Wall'), row=row, col=1)
    #price_graph.add_trace(go.Scatter(x=df.index, y=df['ARIMA_mid_rever'], mode='lines', name='ARIMA Mid Reversed'), row=row, col=1)
    price_graph.add_trace(go.Scatter(x=df.index, y=df['ARIMA_wall_rever'], mode='lines', name='ARIMA Wall Reversed'), row=row, col=1)

    price_graph.add_trace(go.Scatter(x=df.index, y=df['obi'].where(only_obi3), mode='markers', name='Order Book Inbalance 1'), row=row, col=1, secondary_y=True)
    price_graph.add_trace(go.Scatter(x=df.index, y=df['obi3'].where(only_obi3), mode='markers', name='Order Book Inbalance 2'), row=row, col=1, secondary_y=True)
    price_graph.add_trace(go.Scatter(x=df.index, y=df['signal'], mode='lines+markers', name='ratio'), row=row, col=1, secondary_y=True)
    
    price_graph.add_trace(go.Bar(x=df.index, y=df['obi3'], 
            marker_color=[
            'rgba(0, 200, 100, 0.5)' if val >= 0 else 'rgba(255, 50, 50, 0.5)' 
            if pd.notnull(val) else 'rgba(0,0,0,0)' # Transparent for NaNs
            for val in df['obi3']]
            , name='Slope', width=20), row=row, col=1, secondary_y=True,)
    
    #price_graph.add_trace(go.Scatter(x=df.index, y=df['bid_price_1'], mode='lines', name='Best Bid Price'), row=row, col=1)
    #price_graph.add_trace(go.Scatter(x=df.index, y=df['bid_price_2'], mode='lines', name='Level 2 Bid Price'), row=row, col=1)
    price_graph.add_trace(go.Scatter(x=df.index, y=df['bid_price_3'], mode='markers', name='Level 3 Bid Price'), row=row, col=1)

    #price_graph.add_trace(go.Scatter(x=df.index, y=df['ask_price_1'], mode='lines', name='Best Ask Price'), row=row, col=1)
    #price_graph.add_trace(go.Scatter(x=df.index, y=df['ask_price_2'], mode='lines', name='Level 2 Ask Price'), row=row, col=1)
    price_graph.add_trace(go.Scatter(x=df.index, y=df['ask_price_3'], mode='markers', name='Level 3 Ask Price'), row=row, col=1)

price_graph.update_layout(height=800, title_text="Tomato Price Comparison")
price_graph.show()


I don't fully get the maths behind this but using the mid_price ARIMA params for the wall_mid is the best FV. This is probably because l1 has high frequency noise and those params filter out the l1 noise for mid_price, then using them again on wall_mid which is a more stable price indicator is like a double filter.